In [5]:
from google.colab import files

uploaded = files.upload()


Saving archive.zip to archive (1).zip


In [7]:
import pandas as pd
import numpy as np
from pathlib import Path
import zipfile

# 1. Locate and unzip the dataset if necessary
DATA_DIR = Path("/content/olist_data")

# Search for any uploaded zip file in /content
zip_files = list(Path("/content").glob("*.zip"))

if not DATA_DIR.exists() and zip_files:
    zip_path = zip_files[0]
    print(f"Extracting {zip_path} to {DATA_DIR}...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(DATA_DIR)
    print("Extraction complete!")

def load_csv(filename):
    matches = list(DATA_DIR.rglob(filename))
    if not matches:
        # Fallback search inside the main /content directory in case files were extracted elsewhere
        matches = list(Path("/content").rglob(filename))
    if not matches:
        raise FileNotFoundError(
            f"Cannot find {filename}. Please ensure your dataset zip file is uploaded correctly."
        )
    return pd.read_csv(matches[0])

# 2. Load the required datasets
orders = load_csv("olist_orders_dataset.csv")
items = load_csv("olist_order_items_dataset.csv")
products = load_csv("olist_products_dataset.csv")
customers = load_csv("olist_customers_dataset.csv")
sellers = load_csv("olist_sellers_dataset.csv")
translation = load_csv("product_category_name_translation.csv")

print("All required datasets loaded.")

# 3. Convert date columns
date_columns = [
    "order_purchase_timestamp",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_columns:
    orders[col] = pd.to_datetime(
        orders[col], errors="coerce"
    )

# 4. Keep delivered orders with valid timestamps
orders_clean = orders[
    (orders["order_status"] == "delivered") &
    orders["order_purchase_timestamp"].notna() &
    orders["order_delivered_customer_date"].notna()
].copy()

# Calculate actual delivery duration in days
orders_clean["delivery_days"] = (
    orders_clean["order_delivered_customer_date"]
    - orders_clean["order_purchase_timestamp"]
).dt.total_seconds() / 86400

# Remove impossible negative durations
orders_clean = orders_clean[
    orders_clean["delivery_days"] >= 0
].copy()

# 5. Clean product categories and translate to English
products_clean = products[
    ["product_id", "product_category_name"]
].drop_duplicates("product_id")

products_clean = products_clean.merge(
    translation,
    on="product_category_name",
    how="left",
    validate="many_to_one"
)

products_clean["category"] = (
    products_clean["product_category_name_english"]
    .fillna(products_clean["product_category_name"])
    .fillna("unknown")
)

# 6. Link order items with products
items_clean = items.merge(
    products_clean[["product_id", "category"]],
    on="product_id",
    how="left",
    validate="many_to_one"
)

items_clean["category"] = (
    items_clean["category"].fillna("unknown")
)

# 7. Aggregate items to one row per order
def most_common_category(series):
    modes = series.mode()
    return modes.iloc[0] if len(modes) else "unknown"

item_summary = items_clean.groupby("order_id").agg(
    item_count=("order_item_id", "count"),
    total_price=("price", "sum"),
    total_freight=("freight_value", "sum"),
    seller_count=("seller_id", "nunique"),
    category=("category", most_common_category),
    seller_id=("seller_id", "first")
).reset_index()

# 8. Add seller location
seller_info = sellers[
    ["seller_id", "seller_state"]
].drop_duplicates("seller_id")

item_summary = item_summary.merge(
    seller_info,
    on="seller_id",
    how="left",
    validate="many_to_one"
)

# 9. Add customer location
customer_info = customers[
    ["customer_id", "customer_state"]
].drop_duplicates("customer_id")

# 10. Merge into one order-level dataset
df_clean = orders_clean.merge(
    item_summary,
    on="order_id",
    how="inner",
    validate="one_to_one"
)

df_clean = df_clean.merge(
    customer_info,
    on="customer_id",
    how="left",
    validate="many_to_one"
)

# 11. Create purchase-time features
purchase = df_clean["order_purchase_timestamp"]

df_clean["purchase_month"] = purchase.dt.month
df_clean["purchase_day_of_week"] = purchase.dt.dayofweek
df_clean["purchase_hour"] = purchase.dt.hour
df_clean["is_weekend"] = (
    purchase.dt.dayofweek >= 5
).astype(int)

df_clean["same_state"] = (
    df_clean["customer_state"] ==
    df_clean["seller_state"]
).astype(int)

# 12. Calculate the original promised duration
df_clean["estimated_delivery_days"] = (
    df_clean["order_estimated_delivery_date"]
    - df_clean["order_purchase_timestamp"]
).dt.total_seconds() / 86400

# 13. Select final columns
feature_columns = [
    "item_count",
    "total_price",
    "total_freight",
    "seller_count",
    "category",
    "customer_state",
    "seller_state",
    "purchase_month",
    "purchase_day_of_week",
    "purchase_hour",
    "is_weekend",
    "same_state"
]

# Keep only records with valid essential features
df_clean = df_clean.dropna(
    subset=feature_columns + ["delivery_days"]
).copy()

# Remove duplicate order IDs if any
df_clean = df_clean.drop_duplicates(
    subset=["order_id"]
).copy()

# 14. Export a modeling-ready dataset
output_columns = [
    "order_id",
    "order_purchase_timestamp",
    *feature_columns,
    "estimated_delivery_days",
    "delivery_days"
]

processed = df_clean[output_columns].copy()

output_path = Path(
    "/content/preprocessed_delivery_dataset.csv"
)

processed.to_csv(output_path, index=False)

# 15. Print a preprocessing report
print("Preprocessing completed!")
print("Final dataset shape:", processed.shape)
print("Unique orders:", processed["order_id"].nunique())
print("Duplicate orders:", processed["order_id"].duplicated().sum())
print("Missing values:", int(processed.isna().sum().sum()))
print("Saved to:", output_path)

display(processed.head())
display(processed["delivery_days"].describe())

Extracting /content/archive (1).zip to /content/olist_data...
Extraction complete!
All required datasets loaded.
Preprocessing completed!
Final dataset shape: (96470, 16)
Unique orders: 96470
Duplicate orders: 0
Missing values: 0
Saved to: /content/preprocessed_delivery_dataset.csv


,order_id,order_purchase_timestamp,item_count,total_price,total_freight,seller_count,category,customer_state,seller_state,purchase_month,purchase_day_of_week,purchase_hour,is_weekend,same_state,estimated_delivery_days,delivery_days
0,e481f51cbdc54678b7cc49136f2d6af7,2017-10-02 10:56:33,1,29.99,8.72,1,housewares,SP,SP,10,0,10,0,1,15.544063,8.436574
1,53cdb2fc8bc7dce0b6741e2150273451,2018-07-24 20:41:37,1,118.70,22.76,1,perfumery,BA,SP,7,1,20,0,0,19.137766,13.782037
2,47770eb9100c2d0c44946d9cf07ec65d,2018-08-08 08:38:49,1,159.90,19.22,1,auto,GO,SP,8,2,8,0,0,26.639711,9.394213
3,949d5b44dbf5de918fe9c16f97b45f8a,2017-11-18 19:28:06,1,45.00,27.20,1,pet_shop,RN,MG,11,5,19,1,0,26.188819,13.208750
4,ad21c59c0840e6cb83a9ceb5573f8159,2018-02-13 21:18:39,1,19.90,8.72,1,stationery,SP,SP,2,1,21,0,1,12.112049,2.873877


,delivery_days
count,96470.000000
mean,12.558217
std,9.546156
min,0.533414
25%,6.766204
50%,10.217477
75%,15.720182
max,209.628611


In [8]:

from google.colab import files

files.download(
    "/content/preprocessed_delivery_dataset.csv"
)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>